**PART 5.  POSITIONAL ENCODING**

Attention looks at the tokens as a SET: if you shuffle the tokens, the outputs are the same
rows shuffled (I checked it on our example; the vector of a token is the same wherever the
token stands).  So "dog bites man" and "man bites dog" would look alike, the same weakness as
Bag of Words.  The fix is to ADD a position signal to each embedding before the first block:
 
      input of token at position pos  =  embedding  +  PE(pos)
 
The original paper's PE (sinusoidal), for pair index i = 0, 1, ... and model size d:
 
      PE(pos, 2i)   = sin( pos / 10000^(2i / d) )
      PE(pos, 2i+1) = cos( pos / 10000^(2i / d) )
 
Fast pairs (small i) wiggle quickly with pos, slow pairs slowly, like the hands of a clock.
Many modern models instead LEARN a position vector (like an embedding table, GPT-2) or rotate
the Q and K by an angle that depends on position (RoPE, used in Llama).  The idea is the same:
the model must be told where each token stands.
 
  EX 6 (paper, 10 min).  d = 4, so there are two pairs: i = 0 with 10000^(0) = 1, and i = 1 with
  10000^(2/4) = 100.  Write PE(0) and PE(1), each as 4 numbers [sin, cos, sin, cos].  (Use radians:
  sin(1) = 0.8415, cos(1) = 0.5403; for the second pair the angle is pos / 100.)
  And one sentence: why do the same two words in a different order get different vectors only
  AFTER the positions are added?
  * For i = 0: denominator: 10000^0/4 = 1
  * For i = 1: denominator: 10000^1/2 = 100
  * So, angle 0 = 0/1 and 0/100 (both 0), angle 1 = 1/1, 1/100 (1, 0.01)
  * PE(0): [sin 0, cos 0, sin 0, cos 0] = [0, 1, 0, 1]
  * PE(1): [sin 1, cos 1, sin 0.01, cos 0.01] = [0.8415, 0.5403, 0.0100, 1.0000]
  * the same two words in a different order get different vectors only after the positions are added because positions are changed, and meaning are changine, embedding only care about what the word means not the where it is.

**PART 6.  THE BLOCK, THE ENCODER AND THE DECODER**

ONE TRANSFORMER BLOCK  (repeat it N times; the output of one block is the input of the next):
 
   x  ->  Multi-head self-attention  ->  Add & Norm  ->  Feed-forward  ->  Add & Norm  ->  out
 
   - ADD (residual connection):  out = x + sublayer(x).  The original signal is kept, and
     gradients get a shortcut through many layers.
   - NORM (layer normalisation):  each TOKEN's vector is standardised across its own d_model
     numbers, subtract the mean, divide by the std, then a learned scale and shift.  This is
     the z-score of your Day F1 exercise, done on one token at a time.
   - FEED-FORWARD:  Linear(d_model, d_ff) -> activation (ReLU or GELU) -> Linear(d_ff, d_model),
     with d_ff = 4 x d_model in the original paper.  It works on each token ALONE: attention is
     where tokens exchange information, the feed-forward is where each token "thinks".
 
ENCODER AND DECODER
 
   | | Encoder | Decoder |
   |---|---|---|
   | Sees | the whole sequence, left and right | only the tokens to its LEFT (causal mask) |
   | Good for | understanding: classification, search embeddings | generating text, one token at a time |
   | Example | BERT | GPT and Llama (decoder-only) |
   | Both together | T5 and the original translation model: the decoder also attends to the encoder (cross-attention) | |
 
The CAUSAL MASK: before the softmax, set the scores of future tokens to -infinity.  e^(-inf) = 0,
so those weights are exactly 0 and the other weights are renormalised.  Without it the model
could read the answer it is supposed to predict.
 
EX 7 (paper, 10 min).  Take the scaled table of EX 2.
a) Write the table after the causal mask (the entries above the diagonal become -inf).

   S / sqrt(2) = [[0.707, 1.414, 2.121],
                   [0.000, 0.707, 0.707],
                   [0.707, 2.121, 2.828]]
   * row 1 only can see position 1, so position 2 and 3 becomes e^(-inf)
   * row 2 only can see position 1 and 2, so position 3 becomes e^(-inf)
   * row 3 can see all the tokens.
   * so it becomes = [[0.707, e^(-inf), e^(-inf)],
                      [0.000, 0.707, e^(-inf)],
                      [0.707, 2.121, 2.828]]

b) Softmax of row 1 and of row 2.
   * now apply expo of softmax for each row then sum it.
   * then do expo of softmax / sum of expo for each row, then it becomes: 
     row 1 = [1.000, 0.000, 0.000], row 2 = [0.330, 0.670, 0.000]

c) The new vector of token 2 under the mask.
   * New Vector for Token 2 = 0.330 x V_1 + 0.670 x V_2 + 0 x V_3 = 0.330V_1 + 0.670V_2 
     notice how V_3 drops out completely because position 3 is masked out!

d) Why does row 3 not change?
   * Row 3 represents the 3rd token in the sequence. Because there are no tokens in the sequence after position 3, 
     there are no entries above the main diagonal in Row 3.
     
THE WHOLE CHAIN, for a spam classifier you will build in Hour 3 (or for GPT):
 
   text -> tokens -> ids -> embedding + position -> N x [attention, feed-forward]
        -> (classifier) average or first token -> Linear -> spam probability
        -> (GPT) a vector for every position -> Linear to the vocabulary size -> softmax
           -> the probability of the NEXT token

In [5]:
import numpy as np

def softmax(a):
    e = np.exp(a - a.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def attention(Q,K,V, causal=False):
    d_k = Q.shape[1]
    S = Q @ K.T / np.sqrt(d_k)

    if causal:
        mask = np.triu(np.ones(S.shape, dtype=bool), k=1)
        S = np.where(mask, -np.inf, S)
    A = softmax(S)
    return A @ V, A

def positional_encoding(n, d):
    pe = np.zeros((n, d))
    for pos in range(n):
        for i in range(0, d, 2):
            angle = pos / 10000 ** (i/d)
            pe[pos, i] = np.sin(angle)
            pe[pos, i + 1] = np.cos(angle)
    return pe

In [6]:
X  = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)
WQ = np.array([[1, 1], [0, 1]], dtype=float)
WK = np.array([[1, 0], [1, 1]], dtype=float)
WV = np.array([[1, 0], [0, 2]], dtype=float)

O, A = attention(X @ WQ, X @ WK, X @ WV)
print(np.round(A, 4)); print(np.round(O, 4))
Oc, Ac = attention(X @ WQ, X @ WK, X @ WV, causal=True)
print(np.round(Ac, 4)); print(np.round(Oc, 4))
print(np.round(positional_encoding(3, 4), 4))

[[0.14   0.284  0.576 ]
 [0.1978 0.4011 0.4011]
 [0.0743 0.3057 0.62  ]]
[[0.716  1.7199]
 [0.5989 1.6044]
 [0.6943 1.8514]]
[[1.     0.     0.    ]
 [0.3302 0.6698 0.    ]
 [0.0743 0.3057 0.62  ]]
[[1.     0.    ]
 [0.3302 1.3395]
 [0.6943 1.8514]]
[[ 0.      1.      0.      1.    ]
 [ 0.8415  0.5403  0.01    1.    ]
 [ 0.9093 -0.4161  0.02    0.9998]]


In [7]:
p = [2, 0, 1]                                      # a new order of the tokens
O1, _ = attention(X[p] @ WQ, X[p] @ WK, X[p] @ WV)
print(np.allclose(O1, O[p]))                       # (a) predict True or False
pe = positional_encoding(3, 2)
Oa, _ = attention((X + pe) @ WQ, (X + pe) @ WK, (X + pe) @ WV)
Xp = X[p] + pe                                     # shuffle the tokens FIRST, then add the position of each slot
Ob, _ = attention(Xp @ WQ, Xp @ WK, Xp @ WV)
print(np.allclose(Ob, Oa[p]))                      # (b) predict True or False

#(a) is True because standard self-attention without positional encodings is permutation-equivariant (shuffling input tokens simply shuffles the output rows identically), whereas 
#(b) is False because shuffling tokens before adding positional encodings binds the tokens to incorrect slot positions rather than permuting the final outputs.

True
False


**QUESTIONS FOR YOUR NOTES (Q1 to Q8): Day 20**

- **Q1  Write the attention formula and the shape of every object in it.**
    * Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V. Q and K are (n, d_k), V is (n, d_v), the score table Q K^T is (n, n), and the output is (n, d_v).
- **Q2  What do the query, the key and the value do?**
    * The query says what this token is looking for. The key says what each token offers (its label). The value is the content that is passed on, weighted by how well the query matched the key.
- **Q3  Why divide the scores by sqrt(d_k)?**
    * A dot product grows with d_k (its spread is about sqrt(d_k)), which makes the softmax almost one-hot and its gradient nearly zero. Dividing keeps the spread around 1 (softmax([4,0,0]) = 0.965 against softmax([2,0,0]) = 0.787 for the largest weight).
- **Q4  Why several heads instead of one? Does it cost more?**
    * Each head has its own W_Q, W_K, W_V and can look for a different relation. It costs the same as one big attention, because d_head = d_model / h (for example 8 / 2 = 4).
- **Q5  Why does a transformer need positional encoding?**
    * Attention treats the tokens as a set, so shuffling them only shuffles the outputs (my shuffle test: True). Adding a position vector gives each slot a different input, so word order matters. GPT-2 learns the position vectors, and Llama rotates Q and K by position (RoPE).
- **Q6  Name the four parts of a transformer block. What do the residual connection and the layer norm each do?**
    * Multi-head self-attention, add & norm, feed-forward, add & norm. The residual (x + sublayer(x)) keeps the original signal and gives gradients a shortcut. The layer norm standardises each token's vector (a z-score per token) to keep the numbers stable.
- **Q7  Encoder versus decoder: the causal mask, and one model of each kind.**
    * An encoder sees all tokens in both directions (BERT: understanding, classification, embeddings). A decoder uses a causal mask, which sets the scores of future tokens to -infinity before the softmax, so it sees only tokens on its left (GPT and Llama: generating text one token at a time).
- **Q8  The score table is n x n. What does that mean for cost, and why do LLMs have a context window?**
    * Time and memory grow with the square of the length: twice as many tokens means four times the work. So every model has a limit on how many tokens it can read at once (the context window, Day 21).